In [68]:
import numpy as np
from src.ai_lib import layers
from src.ai_lib import Model, metrics, SequenceDataLoader
from src.ai_lib import optimizers
from src.ai_lib import losses
from src.ai_lib import preprocessing

import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text
print(f"Loading completed ! Corpus length : {len(text)} ")

tokenizer = preprocessing.CharTokenizer(text)
data_tensor = np.array(tokenizer.encode(text), dtype=np.int32)


Loading completed ! Corpus length : 1115394 


In [4]:
train_test_split = 0.9

n = int(train_test_split * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

batch_size = 32
block_size = 64  # Numbers of characters to look back at
steps_per_epoch = 100 

# Creating dataloaders
train_loader = SequenceDataLoader(
    data=train_data, 
    block_size=block_size, 
    batch_size=batch_size, 
    steps_per_epoch=steps_per_epoch
)

val_loader = SequenceDataLoader(
    data=val_data, 
    block_size=block_size, 
    batch_size=batch_size, 
    steps_per_epoch=20 # Less steps for validation
)

In [5]:
d_model = 64
n_heads = 4
d_ff = 256

dropout_rate = 0.1

vocab_size = tokenizer.vocab_size

llm_network = layers.Sequential([
    layers.Embedding(vocab_size=vocab_size, d_model=d_model),
    layers.SineCosEncoder(d_model=d_model, max_len=1000),

    layers.Dropout(dropout_rate=dropout_rate),

    layers.TransformerBlock(n_heads=n_heads, d_model=d_model, d_ff=d_ff, is_causal=True),
    layers.Dropout(dropout_rate=dropout_rate),
    layers.TransformerBlock(n_heads=n_heads, d_model=d_model, d_ff=d_ff, is_causal=True),


    layers.LayerNormalization(n_features=d_model),

    layers.Dropout(dropout_rate=dropout_rate),

    layers.Linear(d_model, n_out=vocab_size)
])

model = Model(llm_network)

In [6]:
learning_rate = 1e-4
weight_decay = 1e-3

optimizer = optimizers.Adam(learning_rate=learning_rate, weight_decay=weight_decay)
loss = losses.SoftmaxCrossEntropy()


In [9]:
print("Beginning of training of microgpt")
model.fit(
    dataloader=train_loader,
    epochs=10,
    loss=loss,
    optimizer=optimizer,
    validation_dataloader=val_loader,
    verbose=True,
    metrics=[metrics.accuracy]
)

Beginning of training of microgpt

--- Epoch : 0 ---
Training loss : 1.61703
  accuracy (train) : 0.50862
Validation loss : 1.71662
  accuracy (val) : 0.49429

--- Epoch : 1 ---
Training loss : 1.62419
  accuracy (train) : 0.50865
Validation loss : 1.67414
  accuracy (val) : 0.49353

--- Epoch : 2 ---
Training loss : 1.62383
  accuracy (train) : 0.50860
Validation loss : 1.70401
  accuracy (val) : 0.50232

--- Epoch : 3 ---
Training loss : 1.61912
  accuracy (train) : 0.50910
Validation loss : 1.70133
  accuracy (val) : 0.49119

--- Epoch : 4 ---
Training loss : 1.62046
  accuracy (train) : 0.50856
Validation loss : 1.69429
  accuracy (val) : 0.49224

--- Epoch : 5 ---
Training loss : 1.61209
  accuracy (train) : 0.50971
Validation loss : 1.70073
  accuracy (val) : 0.50029

--- Epoch : 6 ---
Training loss : 1.61741
  accuracy (train) : 0.51086
Validation loss : 1.69039
  accuracy (val) : 0.50339

--- Epoch : 7 ---
Training loss : 1.61827
  accuracy (train) : 0.50923
Validation loss : 1

In [64]:
def generate_text(model, tokenizer, prompt_text="A ", max_new_tokens=100):
    """
    Generates text
    """
    # Important : Turns off setting
    model.sequential.set_training(False)
    
    # Use of kv_cache for inference
    model.sequential.set_use_cache(True)
    model.sequential.reset_cache()
    
    # Encodes context (initial text / prompt)
    context_ids = tokenizer.encode(prompt_text)
    
    # Prefill
    X_input = np.array([context_ids], dtype=np.int32)
    logits = model.predict(X_input)
    
    # We take the predicted char and add it to the list
    next_char_id = sample_top_k(logits[0, -1, :], temperature=1.0, top_k=3)
    context_ids.append(next_char_id)
    
    # Decode
    for _ in range(max_new_tokens - 1):
        # X_input becomes a simple integer as we use kv_cache
        X_input = np.array([[next_char_id]], dtype=np.int32)
        
        logits = model.predict(X_input)
        
        # Since X_input is of length one, logits has shape (1, 1, Vocab_size)
        next_char_id = sample_top_k(logits[0, -1, :], temperature=1.0, top_k=3)
        context_ids.append(next_char_id)
        
    return tokenizer.decode(context_ids)


def sample_top_k(logits, temperature=3.0, top_k=5):
    """
    Sampling from only the k most likely tokens with temperature
    """

    logits = logits / temperature
    
    indices_to_remove = np.argsort(logits)[:-top_k]
    
    logits[indices_to_remove] = -np.inf
    
    exp_preds = np.exp(logits - np.max(logits))
    preds = exp_preds / np.sum(exp_preds)
    
    return np.random.choice(len(logits), p=preds)

generated_text = generate_text(model, tokenizer, prompt_text="KING", max_new_tokens=60)
print(generated_text)

KING EDWARD:
The would not shall be that to this.

LADY ANNE:
Wh


In [66]:
model.save_weights("weights/microgpt_v1.npz")

Model saved : weights/microgpt_v1.npz
